# Customer segmentation with RFM features

Build customer recency, frequency, and monetary features from the uploaded-style transaction data, then evaluate K-Means clustering. Cluster IDs are arbitrary and are not business labels.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if not (root / "datasets").exists(): root = root.parent
df = pd.read_csv(root / "datasets" / "autods_demo_sales.csv", parse_dates=["Order Date"])
required = {"Customer ID", "Order Date", "Revenue"}
missing = required - set(df.columns)
if missing: raise ValueError(f"Customer segmentation requires: {sorted(missing)}")
df = df.dropna(subset=["Customer ID", "Order Date", "Revenue"])
end_date = df["Order Date"].max().normalize()
customers = df.groupby("Customer ID").agg(
    last_purchase=("Order Date", "max"),
    frequency=("Order ID", "nunique") if "Order ID" in df.columns else ("Order Date", "size"),
    monetary=("Revenue", "sum"),
)
customers["recency_days"] = (end_date - customers["last_purchase"].dt.normalize()).dt.days.clip(lower=0)
customers["frequency"] = customers["frequency"].clip(lower=1)
features = pd.DataFrame({
    "recency": np.log1p(customers["recency_days"]),
    "frequency": np.log1p(customers["frequency"]),
    "monetary": np.log1p(customers["monetary"].clip(lower=0)),
})
customers.head()

In [ ]:
if len(features) < 3:
    raise ValueError("At least 3 customers are needed to evaluate clustering.")
k = min(4, len(features) - 1)
scaled = StandardScaler().fit_transform(features)
labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(scaled)
customers["cluster"] = labels
metrics = {
    "customers": len(customers),
    "clusters": k,
    "silhouette_score": silhouette_score(scaled, labels),
    "davies_bouldin_index": davies_bouldin_score(scaled, labels),
}
print(metrics)
display(customers.groupby("cluster").agg(
    customers=("frequency", "size"),
    median_recency_days=("recency_days", "median"),
    median_orders=("frequency", "median"),
    median_revenue=("monetary", "median"),
).round(2))